# Hypothesis Engine -- Training with GRPO (TRL + Unsloth)

This notebook trains a small LLM (Qwen2.5-1.5B) to become a better scientist
using Group Relative Policy Optimization (GRPO) against the Hypothesis Engine
environment.

**Requirements:**
- Google Colab with T4 GPU (free tier works!)
- ~15 minutes for 50 training episodes

**What this demonstrates:**
- Real RL training loop with verifiable reward curves
- Reward improvement over training episodes
- LLM learns to experiment, hypothesize, and predict

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth
!pip install -q "trl>=0.12.0" transformers accelerate bitsandbytes datasets
!pip install -q numpy rich openenv-core==0.2.1
!pip install -q git+https://github.com/AbhinavDubey30/OpenMax.git

In [ ]:
# Step 2: Load model with Unsloth (4-bit quantized for Colab)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: Qwen2.5-1.5B-Instruct (4-bit + LoRA, fp16)")

In [ ]:
# Step 3: Environment-in-the-loop reward function for GRPO
#
# KEY FIXES (vs. old version):
#   - Reward function actually STEPS the environment (not just JSON format check)
#   - Multi-turn context: prompts carry experiment history
#   - Uses real RewardCalculator scores for predictions & hypotheses
#   - Works across all difficulty levels

import json, re, random, copy
from hypothesis_engine import HypothesisEngine

SYSTEM_PROMPT = (
    "You are a scientist investigating a black-box system. "
    "Discover the hidden rule by running experiments, forming hypotheses, "
    "and making predictions.\n\n"
    "Respond with EXACTLY ONE JSON action:\n"
    '- {"action": "experiment", "inputs": {"x": VALUE}}\n'
    '- {"action": "hypothesize", "expression": "MATH_EXPR"}\n'
    '- {"action": "predict", "predictions": [v1, v2, ...]}\n\n'
    "Strategy: Run diverse experiments, find the pattern, hypothesize, predict."
)

# ---- Global store: env_id -> deepcopy of HypothesisEngine after N steps ----
env_store = {}

def extract_json_action(text):
    """Extract a JSON action dict from LLM output."""
    # Try code-fenced JSON first
    for pat in [r'```json\s*(\{.*?\})\s*```', r'```\s*(\{.*?\})\s*```']:
        m = re.search(pat, text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except json.JSONDecodeError: pass
    # Try bare JSON object
    try:
        start = text.index('{')
        depth = 0
        for i in range(start, len(text)):
            if text[i] == '{': depth += 1
            elif text[i] == '}': depth -= 1
            if depth == 0:
                return json.loads(text[start:i+1])
    except (ValueError, json.JSONDecodeError):
        pass
    return None


def hypothesis_reward_fn(completions, prompts, **kwargs):
    """GRPO reward: step the real environment and return actual reward.

    Scoring:
      - Experiments:  env step reward scaled (information gain matters)
      - Hypotheses:   env step reward (accuracy of expression)
      - Predictions:  final env reward (0-100) mapped to [-1, 1]
      - Invalid JSON: -1.0
    """
    rewards = []
    for idx, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        action = extract_json_action(text)

        if action is None:
            rewards.append(-1.0)
            continue

        # --- Look up stored environment snapshot via [ENV:id] marker ---
        prompt = prompts[idx] if idx < len(prompts) else ""
        if isinstance(prompt, list):
            prompt_text = " ".join(m.get("content", "") for m in prompt)
        else:
            prompt_text = str(prompt)

        env_id_match = re.search(r'\[ENV:(\d+)\]', prompt_text)

        if env_id_match:
            eid = int(env_id_match.group(1))
            stored_env = env_store.get(eid)
            if stored_env is not None:
                try:
                    env = copy.deepcopy(stored_env)
                    obs, reward, done, info = env.step(action)

                    if done and "final_reward" in info:
                        # Map final reward 0-100 -> [-1, 1]
                        total = info["final_reward"].get("total_reward", 0)
                        rewards.append(total / 50.0 - 1.0)
                    else:
                        rewards.append(max(-1.0, min(2.0, reward / 5.0)))
                    continue
                except Exception:
                    pass  # fall through to format-based fallback

        # Fallback: lightweight format-based reward (still better than nothing)
        act = action.get("action", "")
        if act == "predict":   rewards.append(0.3)
        elif act == "experiment": rewards.append(0.1)
        elif act == "hypothesize": rewards.append(0.2)
        else: rewards.append(-0.5)

    return rewards

print("Environment-in-the-loop reward function defined.")

In [ ]:
# Step 4: Generate MULTI-TURN prompts with experiment history
#
# KEY FIXES:
#   - Prompts include real experiment history (not static one-shot)
#   - All difficulty levels 1-7 (not just 1-3)
#   - No category / hints leaked in the prompt
#   - Environment snapshot stored so reward fn can step the REAL env

from datasets import Dataset

def generate_training_prompts(n_prompts=100):
    """Generate multi-turn prompts with experiment history from diverse worlds."""
    prompts = []
    difficulty_counts = {}

    for i in range(n_prompts):
        difficulty = random.choice(range(1, 8))
        difficulty_counts[difficulty] = difficulty_counts.get(difficulty, 0) + 1
        env = HypothesisEngine(difficulty=difficulty, experiment_budget=15)
        obs = env.reset()
        world = obs.get("world", {})

        variables = world.get("variables", ["x"])
        ranges = world.get("ranges", {v: (-10, 10) for v in variables})
        n_test = len(obs.get("test_cases", []))

        # ---- Play 0-6 heuristic experiment steps to build history ----
        max_heuristic = min(6, obs.get("experiments_remaining", 6) - 2)
        n_steps = random.randint(0, max(0, max_heuristic))
        history_lines = []

        for s in range(n_steps):
            var = variables[0]
            lo, hi = ranges.get(var, (-10, 10))
            val = round(lo + (hi - lo) * (s / max(n_steps, 1)) + random.uniform(-0.5, 0.5), 2)
            action = {"action": "experiment", "inputs": {var: val}}

            obs, _, done, _ = env.step(action)
            if "output" in obs:
                history_lines.append(
                    f"  x={val} -> output={obs['output']}"
                )
            if done:
                break

        # ---- Build prompt (NO category, NO hints, NO world_type) ----
        parts = [
            f"Black-box system with input variables: {variables}",
            f"Ranges: {json.dumps({v: list(ranges[v]) for v in variables})}",
            f"Experiment budget remaining: {obs.get('experiments_remaining', 0)}",
            f"Test cases to predict: {n_test}",
        ]
        if history_lines:
            parts.append("Experiment results so far:")
            parts.extend(history_lines)
        parts.append("What is your next action?")
        # Hidden env-id marker (survives tokenization, used by reward fn)
        parts.append(f"[ENV:{i}]")

        user_msg = "\n".join(parts)

        # Store env snapshot for the reward function
        env_store[i] = copy.deepcopy(env)

        prompts.append({"prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]})

    return prompts, difficulty_counts

training_prompts, diff_dist = generate_training_prompts(100)
train_dataset = Dataset.from_list(training_prompts)
with_history = sum(1 for p in training_prompts if "results so far" in p["prompt"][1]["content"])
print(f"Generated {len(training_prompts)} multi-turn training prompts")
print(f"  Difficulty distribution: {json.dumps(dict(sorted(diff_dist.items())))}")
print(f"  Prompts with experiment history: {with_history}/{len(training_prompts)}")
print(f"  Environment snapshots stored: {len(env_store)}")

In [ ]:
# Step 5: GRPO Training
from trl import GRPOConfig, GRPOTrainer
import types

training_args = GRPOConfig(
    output_dir="./hypothesis_engine_grpo",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    max_prompt_length=1024,      # longer: prompts now carry experiment history
    max_completion_length=256,
    num_generations=4,
    logging_steps=5,
    save_steps=50,
    warmup_steps=10,
    fp16=True,
    report_to="none",
)

FastLanguageModel.for_training(model)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    reward_funcs=hypothesis_reward_fn,
)

# ---- Monkey-patch: fix Unsloth compiled-cache completion-mask size bug ----
# Unsloth's compiled GRPOTrainer can produce per-token tensors with a
# different seq-len than completion_mask.  Pad all 2-D tensors in the
# prepared batch to the same length so compute_loss never crashes.
# Zero-padding is safe: completion_mask is 0 for padded positions,
# so masked products are unaffected.
_orig_prepare = trainer._prepare_inputs

def _safe_prepare(self, inputs):
    result = _orig_prepare(inputs)
    keys_2d = [k for k, v in result.items()
               if isinstance(v, torch.Tensor) and v.dim() == 2]
    if keys_2d:
        max_len = max(result[k].shape[1] for k in keys_2d)
        for k in keys_2d:
            t = result[k]
            if t.shape[1] < max_len:
                pad = torch.zeros(t.shape[0], max_len - t.shape[1],
                                  dtype=t.dtype, device=t.device)
                result[k] = torch.cat([t, pad], dim=1)
    return result

trainer._prepare_inputs = types.MethodType(_safe_prepare, trainer)
# ---------------------------------------------------------------------------

print("Starting GRPO training (env-in-the-loop rewards)...")
trainer.train()
print("Training complete!")

In [ ]:
# Step 6: Evaluate and visualize results
import matplotlib.pyplot as plt

def run_eval_episode(model, tokenizer, difficulty=1, max_steps=12):
    """Run one eval episode and return final reward."""
    env = HypothesisEngine(difficulty=difficulty, experiment_budget=max_steps)
    obs = env.reset()
    world = obs.get("world", {})
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"World: {world.get('world_name','?')}\nDescription: {world.get('description','')}\nVariables: {world.get('variables',[])}\nBudget: {obs.get('experiments_remaining',0)}\nBegin!"},
    ]
    
    done = False
    step = 0
    total_reward = 0.0
    
    FastLanguageModel.for_inference(model)
    while not done and step < max_steps + 5:
        input_ids = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(input_ids=input_ids, max_new_tokens=256, temperature=0.7, do_sample=True)
        response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
        messages.append({"role": "assistant", "content": response})
        
        action = extract_json_action(response)
        if action is None:
            action = {"action": "experiment", "inputs": {"x": float(step)}}
        
        obs, reward, done, info = env.step(action)
        total_reward += reward
        step += 1
        
        if done:
            return info.get("final_reward", {}).get("total_reward", total_reward)
        messages.append({"role": "user", "content": obs.get("message", "")[:200]})
    
    return total_reward

# Run evaluation
print("Evaluating trained model on 10 episodes...")
rewards = []
for i in range(10):
    r = run_eval_episode(model, tokenizer, difficulty=1)
    rewards.append(r)
    print(f"  Episode {i+1}: {r:.1f}/100")

avg_reward = sum(rewards) / len(rewards)
print(f"\nAverage trained reward: {avg_reward:.1f}/100")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, 11), rewards, color="#4c6ef5", alpha=0.8)
ax.axhline(y=avg_reward, color="red", linestyle="--", label=f"Mean: {avg_reward:.1f}")
ax.set_xlabel("Episode")
ax.set_ylabel("Reward (0-100)")
ax.set_title("Hypothesis Engine: Post-Training Evaluation")
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Results saved to training_results.png")

In [ ]:
# Step 7: Save the trained model
model.save_pretrained("hypothesis_scientist_lora")
tokenizer.save_pretrained("hypothesis_scientist_lora")
print("Trained model saved to hypothesis_scientist_lora/")
print("\n=== Training Summary ===")
print(f"  Model: Qwen2.5-1.5B-Instruct + LoRA (4-bit)")
print(f"  Method: GRPO (Group Relative Policy Optimization)")
print(f"  Environment: Hypothesis Engine v2.0")
print(f"  Training prompts: {len(training_prompts)}")
print(f"  Average eval reward: {avg_reward:.1f}/100")